# Cytoscape CX2 export

CX2 is useful when a graph should be inspected in Cytoscape.
AnnNet can choose how hyperedges are projected for tools that
do not support them directly.


In [ ]:
import annnet as an

an.info()


## Build a graph with a hyperedge


In [ ]:
G = an.AnnNet(directed=True)
G.add_vertices(['A', 'B', 'C', 'D', 'E'])
G.add_edges('A', 'B', edge_id='binary_edge', weight=1.0)
G.add_edges('A', 'B', edge_id='parallel_edge', weight=0.65)
G.add_edges('B', 'C', edge_id='complex_edge', directed=False, weight=0.8)
G.add_edges(src=['B', 'C'], tgt=['D', 'E'], edge_id='reaction', directed=True, weight=2.0)
G.attrs.set_vertex_attrs_bulk(
    {
        'A': {'label': 'ligand'},
        'B': {'label': 'enzyme'},
        'C': {'label': 'cofactor'},
        'D': {'label': 'product'},
        'E': {'label': 'byproduct'},
    }
)

G.views.edges().select(['edge_id', 'kind', 'source', 'target', 'head', 'tail'])


## Preview the network

The CX2 file is for Cytoscape. For static documentation, a
Graphviz preview gives a lightweight embedded view of the same
topology.


In [ ]:
from annnet.utils import plotting

plotting.plot(G, backend='graphviz', show_edge_labels=True)


## Compare hyperedge export modes


In [ ]:
for mode in ['skip', 'expand', 'reify']:
    cx2 = an.to_cx2(G, hyperedges=mode, export_name=f'annnet-{mode}')
    edge_aspect = next(item['edges'] for item in cx2 if 'edges' in item)
    node_aspect = next(item['nodes'] for item in cx2 if 'nodes' in item)
    print(mode, 'nodes:', len(node_aspect), 'edges:', len(edge_aspect))


## Round-trip through the CX2 manifest


In [ ]:
import json

cx2 = an.to_cx2(G, hyperedges='reify', export_name='annnet-reified')
restored = an.from_cx2(cx2)

print('restored shape:', restored.shape)
print('restored hyperedges:', sorted(restored.hyperedge_definitions))
print(json.dumps(cx2[:3], indent=2))


`skip`, `expand`, and `reify` are explicit choices about what
Cytoscape should see. A true embedded Cytoscape.js widget would
need an additional browser asset; this notebook keeps the docs
static by showing the CX2 payload plus a Graphviz preview.
